# MetroPT-3 — Exploratory Data Analysis

**PdM-Agent · AI 894 capstone, Group 6 (Project 06)**
Rosh Chathoth Sreedharan · Ashok Reddy Buthukuri · Gayatri Suresh Chaudhari

This notebook is our first structured pass over the MetroPT-3 dataset before any modeling work.
It has five jobs:

1. Settle the **sampling rate** from the file's own timestamps (our sources disagree between 0.1 Hz and 1 Hz).
2. Map **coverage and gaps** across the seven-month recording period.
3. Characterize each of the 15 signals, and how they behave in **healthy vs. failure** periods.
4. Identify **healthy windows** suitable for fitting the anomaly detector.
5. Produce the facts that go into the **asset profile YAML** (signals, units, sampling interval, failure labels).

Everything downstream — detector interface, MCP diagnostic checks, evaluation — reads from what this notebook establishes.

The section notes fold in what the first full run (September 2026) established, so the prose
states findings rather than expectations; re-running reproduces every number.

## Dataset notes

MetroPT-3 (UCI ML Repository, dataset 791, CC BY 4.0) contains readings from the Air Production Unit (APU)
compressor of a Metro do Porto train, recorded February–August 2020: 7 analog and 8 digital channels,
1,516,948 records. Failure information comes from the operator's maintenance reports as tabulated in the
accompanying paper (Veloso et al., 2022, *The MetroPT dataset for predictive maintenance*).

Two numbers about this dataset circulate in our own documents — 1,516,948 records and 15.2 M readings.
The load cell below checks the file itself; 1,516,948 is the row count we hold ourselves to.

### Signal dictionary

Descriptions paraphrased from the dataset paper. Polarity of the digital channels (what "1" means)
should be verified against the paper before we hard-code it anywhere.

| Signal | Type | Unit | Description (per Veloso et al., 2022) |
|---|---|---|---|
| TP2 | analog | bar | Pressure at the compressor |
| TP3 | analog | bar | Pressure at the pneumatic panel |
| H1 | analog | bar | Pressure drop at the cyclonic separator filter |
| DV_pressure | analog | bar | Pressure drop at the towers' discharge |
| Reservoirs | analog | bar | Downstream reservoir pressure (tracks TP3) |
| Oil_temperature | analog | °C | Compressor oil temperature |
| Motor_current | analog | A | Current of one motor phase (~0 off, ~4 A off-load, ~7 A under load) |
| COMP | digital | 0/1 | Air-intake valve signal; active when there is no air intake |
| DV_eletric | digital | 0/1 | Outlet valve signal; active when compressor runs under load |
| Towers | digital | 0/1 | Which air-drying tower is in service |
| MPG | digital | 0/1 | Starts the compressor under load when pressure < 8.2 bar |
| LPS | digital | 0/1 | Low-pressure switch; activates below 7 bar |
| Pressure_switch | digital | 0/1 | Detects discharge in the drying towers |
| Oil_level | digital | 0/1 | Active when oil level is below expected |
| Caudal_impulses | digital | 0/1 | Flowmeter (airflow) pulse signal |

(The spellings `DV_eletric` and `Caudal_impulses` are the dataset's own.)

## 0 · Setup

Built to run in **Google Colab**: the data cell below fetches MetroPT-3 straight from the UCI
archive on first run (a one-time download of roughly 60 MB), so *Runtime → Run all* is enough.
Running locally works the same way; if you already have `MetroPT3(AirCompressor).csv`, put it next
to the notebook (or under `data/`) and the download is skipped.

The failure windows are entered by hand from the paper — verify them there before treating them as
evaluation labels.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

# --- data location: edit if your copy lives elsewhere -------------------
_CANDIDATES = [
    Path('MetroPT3(AirCompressor).csv'),
    Path('data/MetroPT3(AirCompressor).csv'),
    Path('../data/MetroPT3(AirCompressor).csv'),
    Path('/content/MetroPT3(AirCompressor).csv'),
]

EXPECTED_RECORDS = 1_516_948   # UCI dataset page (not 15.2 M)

ANALOG = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
          'Oil_temperature', 'Motor_current']
DIGITAL = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
           'Pressure_switch', 'Oil_level', 'Caudal_impulses']
UNITS = {'TP2': 'bar', 'TP3': 'bar', 'H1': 'bar', 'DV_pressure': 'bar',
         'Reservoirs': 'bar', 'Oil_temperature': 'degC', 'Motor_current': 'A'}

# Failure windows from the maintenance reports as tabulated in
# Veloso et al. (2022). Entered by hand -- VERIFY against the paper
# before using these as evaluation labels.
FAILURE_WINDOWS = [
    ('2020-04-18 00:00', '2020-04-18 23:59', 'Air leak 1'),
    ('2020-05-29 23:30', '2020-05-30 06:00', 'Air leak 2'),
    ('2020-06-05 10:00', '2020-06-07 14:30', 'Air leak 3'),
    ('2020-07-15 14:30', '2020-07-15 19:00', 'Air leak 4'),
]
FAILURE_WINDOWS = [(pd.Timestamp(s), pd.Timestamp(e), n)
                   for s, e, n in FAILURE_WINDOWS]

In [ ]:
# Get the data. In Colab (or anywhere with internet) this downloads the
# official zip from UCI on first run; it is skipped once the CSV exists.
import io
import zipfile
import urllib.request

if not any(p.exists() for p in _CANDIDATES):
    print('Downloading MetroPT-3 from the UCI archive ...')
    req = urllib.request.Request(
        'https://archive.ics.uci.edu/static/public/791/metropt+3+dataset.zip',
        headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r:
        buf = io.BytesIO(r.read())
    data_dir = Path('data')
    data_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(buf) as z:
        member = next(m for m in z.namelist() if m.endswith('.csv'))
        extracted = Path(z.extract(member, data_dir))
    target = data_dir / 'MetroPT3(AirCompressor).csv'
    if extracted != target:
        extracted.rename(target)
    print(f'Saved {target}')

DATA_PATH = next((p for p in _CANDIDATES if p.exists()), _CANDIDATES[0])
print(f'Using: {DATA_PATH}')

In [ ]:
# Chart style: one restrained palette, hairline grid, red reserved for failures.
SURFACE, GRID = '#fcfcfb', '#e1e0d9'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
CRITICAL = '#d03b3b'
DIVERGING = LinearSegmentedColormap.from_list(
    'div', ['#2a78d6', '#f0efec', '#d03b3b'])

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.edgecolor': '#c3c2b7', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.labelcolor': INK2, 'text.color': INK,
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'figure.dpi': 110,
    'axes.prop_cycle': cycler(color=[BLUE, ORANGE, AQUA, YELLOW]),
})

def shade_failures(ax):
    # Red wash over the reported failure windows.
    for s, e, _ in FAILURE_WINDOWS:
        ax.axvspan(s, e, color=CRITICAL, alpha=0.18, lw=0)

## 1 · Load and first checks

Record count against the UCI figure, time span, and schema.

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

match = 'match' if len(df) == EXPECTED_RECORDS else 'MISMATCH -- investigate'
print(f'File:    {DATA_PATH}')
print(f'Records: {len(df):,}  (expected {EXPECTED_RECORDS:,} -> {match})')
print(f'Span:    {df.timestamp.min()}  ->  {df.timestamp.max()}')
print(f'Memory:  {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB')
missing_cols = [c for c in ANALOG + DIGITAL if c not in df.columns]
print('Schema:  all 15 expected signals present' if not missing_cols
      else f'Schema:  MISSING {missing_cols} -- continuing with what is present')

# If the schema ever drifts, analyze the columns that exist rather than crash.
ANALOG = [c for c in ANALOG if c in df.columns]
DIGITAL = [c for c in DIGITAL if c in df.columns]
df.head()

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'nulls': df.isna().sum(),
    'unique': df.nunique(),
})
summary

In [ ]:
# Integrity: digital channels binary? constant columns? duplicate timestamps?
issues = []
for c in DIGITAL:
    vals = set(df[c].dropna().unique())
    if not vals <= {0, 1}:
        issues.append((c, sorted(vals)[:10]))
print('All digital channels are strictly binary (0/1).' if not issues
      else f'Non-binary digital channels: {issues}')

const = [c for c in ANALOG + DIGITAL if df[c].nunique() <= 1]
print('Constant columns:', const if const else 'none')

dup_ts = int(df.timestamp.duplicated().sum())
print(f'Duplicated timestamps: {dup_ts:,}')

df[ANALOG].describe().T.round(2)

## 2 · Sampling rate and coverage

Our sources disagreed between 1 Hz and 0.1 Hz. The timestamps settle it: the median delta is
10 s — an effective rate of **0.1 Hz** — with real jitter around it (deltas of 9–13 s are
common), so windowing code should resample to a regular 10 s grid rather than assume one.
This number goes straight into the asset profile YAML, and the proposal's 1 Hz mention gets
corrected to match.

Coverage is the other headline: about 82 % of the span is recorded, across 331 gaps totalling
roughly 38 days. Several gaps exceed a full day, and one falls immediately after the third
failure window — repair downtime, most likely. Windows that straddle a gap should be dropped,
not bridged.

In [ ]:
dt = df.timestamp.diff().dt.total_seconds().dropna()
step = float(dt.median())

print('Most common timestamp deltas (seconds):')
print(dt.value_counts().head(8).rename('count').to_frame())
print()
print(f'Median delta: {step:g} s  ->  effective sampling rate {1 / step:.3g} Hz')

span_s = (df.timestamp.max() - df.timestamp.min()).total_seconds()
print(f'Signal time: {len(df):,} records x {step:g} s '
      f'= {len(df) * step / 86400:,.1f} days over a '
      f'{span_s / 86400:,.1f}-day span ({len(df) * step / span_s:.1%} coverage)')

In [ ]:
# Gaps: any delta well beyond the median step.
GAP_FACTOR = 5
g = pd.DataFrame({
    'gap_start': df.timestamp.shift(1),
    'gap_end': df.timestamp,
    'delta_s': df.timestamp.diff().dt.total_seconds(),
}).dropna()
big = g[g.delta_s > GAP_FACTOR * step].sort_values('delta_s', ascending=False)

print(f'{len(big)} gaps longer than {GAP_FACTOR}x the median step; '
      f'{big.delta_s.sum() / 3600:,.1f} h missing in total')
print('Ten longest:')
(big.head(10)
    .assign(duration=lambda x: pd.to_timedelta(x.delta_s, unit='s'))
    [['gap_start', 'gap_end', 'duration']]
    .reset_index(drop=True))

In [ ]:
ts = df.set_index('timestamp')
daily = ts.resample('D').size()
expected_daily = 86400 / step

fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_between(daily.index, daily / expected_daily * 100,
                step='mid', color=BLUE, alpha=0.85, lw=0)
shade_failures(ax)
ax.set_ylabel('% of expected records')
ax.set_ylim(0, 112)
ax.set_title('Daily coverage -- dips are recording gaps; red = reported failure windows')
ax.grid(axis='x', visible=False)
plt.show()

## 3 · Analog signals

Distributions first, then the whole period at a glance, then one hour of raw signal, and finally
the correlation structure. Three things stand out on the full data:

- Normal operation is **short load bursts** — the compressor is under load only ~16 % of the time
  (the hour of raw signal below shows a single two-minute charge in an hour). That is what makes
  TP2 and Motor_current strongly bimodal, plain thresholds useless, and reconstruction-based
  detection attractive.
- **February is a different operating regime**: light duty throughout, LPS silent, and the shift
  happens across the Feb 28 – Mar 1 gap. The detector should not be fit on February alone, and
  train/validation splits should respect the regime change.
- **TP3 and Reservoirs are duplicates** (r = 1.00): one of them suffices as a detector input, and
  their divergence becomes a free sensor-health check. TP2 and H1 mirror each other (r = −0.96),
  so H1 adds little linear information — but its failure behavior (Section 5) is the crispest
  signature in the dataset.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
for ax, c in zip(axes.flat, ANALOG):
    ax.hist(df[c].dropna(), bins=60, color=BLUE, lw=0)
    ax.set_title(f'{c} ({UNITS[c]})', fontsize=9.5)
    ax.set_yticks([])
    ax.grid(axis='x', visible=False)
axes.flat[-1].axis('off')
fig.suptitle('Analog signal distributions, full period', x=0.01, ha='left',
             fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
res = ts[ANALOG].resample('10min').mean()
fig, axes = plt.subplots(len(ANALOG), 1, figsize=(11, 10), sharex=True)
for ax, c in zip(axes, ANALOG):
    ax.plot(res.index, res[c], color=BLUE, lw=0.6)
    shade_failures(ax)
    ax.set_ylabel(f'{c}\n({UNITS[c]})', rotation=0, ha='right',
                  va='center', fontsize=8.5)
    ax.grid(axis='x', visible=False)
axes[0].set_title('Seven months at a glance (10-minute means; red = reported failure windows)')
fig.align_ylabels(axes)
plt.show()

In [ ]:
# One hour of raw signal: pick an early hour with near-complete coverage.
hourly = ts.resample('h').size()
full_hours = hourly[hourly >= 0.95 * 3600 / step]
zoom_start = (full_hours.index[min(48, len(full_hours) - 1)]
              if len(full_hours) else ts.index[0])
zoom = ts.loc[zoom_start: zoom_start + pd.Timedelta(hours=1)]

fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 3, 1]})
axes[0].plot(zoom.index, zoom.TP2, color=BLUE, lw=1.2)
axes[0].set_ylabel('TP2 (bar)')
axes[1].plot(zoom.index, zoom.Motor_current, color=ORANGE, lw=1.2)
axes[1].set_ylabel('Motor_current (A)')
axes[2].step(zoom.index, zoom.COMP, where='post', color=AQUA, lw=1.2)
axes[2].set_ylabel('COMP')
axes[2].set_yticks([0, 1])
axes[0].set_title(f'One hour of raw signal, starting {zoom_start}')
for ax in axes:
    ax.grid(axis='x', visible=False)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.show()

In [ ]:
corr = ts[ANALOG].corr()
n = len(ANALOG)
fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(corr, cmap=DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(n), ANALOG, rotation=45, ha='right', fontsize=8.5)
ax.set_yticks(range(n), ANALOG, fontsize=8.5)
for i in range(n):
    for j in range(n):
        v = corr.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='#0b0b0b' if abs(v) < 0.65 else '#ffffff')
ax.set_title('Pearson correlation, analog signals')
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

## 4 · Digital signals

Duty cycles over the whole period first, then the cycling story. Three findings:

- **Channel polarity needs verifying before any diagnostic check uses it.** Oil_level reads
  active 90.4 % of the time and Pressure_switch 99.1 % — hard to reconcile with the paper's
  "active when oil is below expected". Until checked against the paper, the digital
  descriptions in the signal dictionary are treated as unconfirmed.
- **Cycling rate is the strongest precursor available.** COMP cycles sit near 50 per day in
  normal operation and spike to 750–1,400 around failures — including a ~1,400-cycle spike on
  May 27–28, one to two days *before* the reported second leak. Hours-to-days of warning looks
  realistic, which is what the lead-time success metric needs.
- **Not every anomaly is labeled.** Cycling spikes near Jun 13 and Jul 25, and heavy LPS days
  (Apr 25 — followed by a two-day gap — early May, Jul 1), have no reported failure. Under the
  current labels these will score as false alarms; evaluation has to decide how to treat them
  before any FAR-per-week number is quoted.

One caveat stands: at a 10 s sampling interval, cycles shorter than a couple of samples alias,
so cycle counts are a lower bound.

In [ ]:
duty = ts[DIGITAL].mean().sort_values() * 100
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(duty.index, duty, color=BLUE, height=0.62)
for y, v in enumerate(duty):
    ax.text(v + 1.2, y, f'{v:.1f}%', va='center', fontsize=8.5, color=INK2)
ax.set_xlabel('% of samples active (= 1)')
ax.set_xlim(0, 112)
ax.set_title('Digital channel duty cycles, full period')
ax.grid(axis='y', visible=False)
plt.show()

In [ ]:
comp_cycles = (ts.COMP.diff().abs() == 1).resample('D').sum() / 2
lps_hours = ts.LPS.resample('D').mean() * 24

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(comp_cycles.index, comp_cycles, color=BLUE, lw=1.1)
axes[0].set_ylabel('COMP cycles / day')
axes[0].set_title('Compressor cycling rate and low-pressure alarms '
                  '(red = reported failure windows)')
axes[1].plot(lps_hours.index, lps_hours, color=ORANGE, lw=1.1)
axes[1].set_ylabel('LPS active (h / day)')
for ax in axes:
    shade_failures(ax)
    ax.grid(axis='x', visible=False)
plt.show()

## 5 · Failure windows

Each reported failure, plus and minus three days, on the three most diagnostic signals. The
pattern is the same in all four windows: **normal cycling stops and the unit runs continuously
under load** — TP2 pinned near 8–9 bar, Motor_current flat around 5.6 A, oil temperature
climbing to 75–90 °C (the fourth leak reaches ~89 °C, the dataset maximum), and H1 collapsed to
~0 for the whole window. Detecting a failure *while it is happening* is therefore easy; the
modeling problem worth solving is the precursor phase in the days before.

Two label caveats. The first window spans exactly 00:00–23:59, so the reports are day-granular —
window edges are not precise timestamps, and evaluation should use guard bands around them.
And because TP2, H1 and DV_pressure idle near zero, percentage change against baseline is
misleading; the comparison below uses absolute differences in each signal's own unit.

In [ ]:
ZOOM_PAD = pd.Timedelta(days=3)
SIG3 = ['TP2', 'Motor_current', 'Oil_temperature']

for s, e, name in FAILURE_WINDOWS:
    seg = ts.loc[s - ZOOM_PAD: e + ZOOM_PAD, SIG3].resample('1min').mean()
    if seg.dropna(how='all').empty:
        print(f'{name}: no data within +/- 3 days of the window')
        continue
    fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
    for ax, c in zip(axes, SIG3):
        ax.plot(seg.index, seg[c], color=BLUE, lw=0.8)
        ax.axvspan(s, e, color=CRITICAL, alpha=0.18, lw=0)
        ax.set_ylabel(f'{c}\n({UNITS[c]})', rotation=0, ha='right',
                      va='center', fontsize=8.5)
        ax.grid(axis='x', visible=False)
    axes[0].set_title(f'{name}: {s:%Y-%m-%d %H:%M} -> {e:%Y-%m-%d %H:%M} '
                      f'(+/- 3 days, 1-minute means)')
    fig.align_ylabels(axes)
    plt.show()

In [ ]:
BASELINE = pd.Timedelta(days=7)
rows = {}
for s, e, name in FAILURE_WINDOWS:
    during = ts.loc[s:e, ANALOG].mean()
    before = ts.loc[s - BASELINE: s, ANALOG].mean()
    rows[name] = during - before
chg = pd.DataFrame(rows).loc[ANALOG].round(2)
print('Mean level during each failure window minus the preceding 7 days')
print('(absolute change, in each signal unit -- bar, degC, A):')
chg

## 6 · Healthy windows for detector training

The detector interface fits on healthy data only, so we need spans that are (a) well covered and
(b) clear of any reported failure with a safety buffer. Candidate days: at least 90 % of expected
records, at least two days away from every failure window.

The day-level view is sobering: only 80 days clear the bar, and the longest contiguous run is
Jun 29 – Jul 5 — seven days. Two consequences for the design: healthy data should be selected at
*window* level rather than day level, and the flat "parked" stretches visible in Section 5
(e.g. Apr 17, May 27) are idle rather than representative-healthy, so fitting should exclude
them explicitly.

In [ ]:
FAIL_BUFFER = pd.Timedelta(days=2)
cov = ts.resample('D').size() / (86400 / step)

bad_days = set()
for s, e, _ in FAILURE_WINDOWS:
    rng = pd.date_range((s - FAIL_BUFFER).normalize(),
                        (e + FAIL_BUFFER).normalize(), freq='D')
    bad_days.update(rng.date)

ok_dates = sorted(d.date() for d, c in cov.items()
                  if c >= 0.90 and d.date() not in bad_days)

runs, start, prev = [], None, None
for d in ok_dates:
    if start is None:
        start = prev = d
    elif (d - prev).days == 1:
        prev = d
    else:
        runs.append((start, prev))
        start = prev = d
if start is not None:
    runs.append((start, prev))

runs_df = (pd.DataFrame(runs, columns=['start', 'end'])
             .assign(days=lambda x: (pd.to_datetime(x.end)
                                     - pd.to_datetime(x.start)).dt.days + 1)
             .sort_values('days', ascending=False)
             .reset_index(drop=True))
print(f'{len(ok_dates)} candidate healthy days '
      f'(>= 90% coverage, >= 2 days clear of any reported failure)')
print('Longest contiguous healthy spans:')
runs_df.head(10)

## 7 · Fact sheet and implications

In [ ]:
print('EDA fact sheet')
print('--------------')
print(f'records            : {len(df):,}')
print(f'span               : {df.timestamp.min():%Y-%m-%d} -> {df.timestamp.max():%Y-%m-%d}')
print(f'sampling step      : {step:g} s  ({1 / step:.3g} Hz)')
print(f'gaps > {GAP_FACTOR}x step     : {len(big):,}  ({big.delta_s.sum() / 3600:,.1f} h)')
print(f'duplicated stamps  : {dup_ts:,}')
fail_h = sum((e - s).total_seconds() for s, e, _ in FAILURE_WINDOWS) / 3600
print(f'reported failures  : {len(FAILURE_WINDOWS)} windows, {fail_h:.1f} h total')
longest = int(runs_df.days.max()) if len(runs_df) else 0
print(f'healthy candidates : {len(ok_dates)} days; longest contiguous span {longest} days')

### What the first full run established

1. **Sampling rate: 0.1 Hz** — median delta 10 s, jittered 9–13 s. The proposal's 1 Hz mention
   is corrected accordingly.
2. **Record count: 1,516,948** — exact match to the UCI figure.
3. **Clean file, gappy recording** — no nulls, no duplicate timestamps, strictly binary digitals,
   but ~38 days missing across 331 gaps; one gap directly follows the third failure.
4. **Failure signature: continuous load replaces cycling.** During-failure detection is easy;
   the objective that earns the project its keep is early warning.
5. **Lead time exists** — the cycling rate spikes one to two days before at least one reported
   failure.
6. **Open items** — digital-channel polarity; day-granular failure labels (guard-band rule);
   the unlabeled episodes (Jun 13, Jul 25, heavy LPS days); TP3 vs. Reservoirs as the retained
   input.

### What this feeds in PdM-Agent

- **Asset profile YAML** — the 15 signals with units, the 10 s step, window-level healthy spans
  (Section 6), the failure windows with guard bands as evaluation labels, and one of
  TP3/Reservoirs as detector input with their divergence kept as a sensor-health check.
- **Detector interface** — fit on window-level healthy segments drawn from both operating
  regimes (not February alone); score per-signal reconstruction error on the analog channels;
  Section 4 justifies a rolling COMP-cycle-rate feature as an engineered channel.
- **Gap policy** — resample to a regular 10 s grid; drop windows straddling more than a few
  missing samples; never bridge the day-scale gaps.
- **Evaluation** — lead time and false-alarm rate need the verified windows plus a decision on
  the unlabeled episodes; report FAR both with and without those periods excluded, and say so.

### Action plan

1. **First action (2 minutes, today):** paste the fact sheet above into the proposal and change
   "1 Hz" to "0.1 Hz (10 s, jittered)". Same sitting, before anything else.
2. **If** we agree the labels are day-granular, **then** we fix the guard-band rule at the next
   team sync, before any metric code exists.
3. **Version Zero of `asset_profile.yaml` today:** signals and units, 10 s step, the
   TP3-or-Reservoirs choice, the four windows, Jun 29 – Jul 5 as the first fit window. Ugly is
   fine; complete beats polished.
4. **Hard thing first:** the gap/window policy — it constrains every window the detector ever
   sees, so it precedes any model code.
5. **Accountability:** executed notebook and fact sheet go to Ashok and Gayatri before the next
   sync, with the open items named — digital polarity, and the unlabeled Jun 13 / Jul 25
   episodes.